In [32]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [33]:
#Load the IMDB dataset
word_index = imdb.get_word_index()
rev_word_index = {value:key for key, value in word_index.items()}

In [34]:
#Load the Model
model = tf.keras.models.load_model('simple_rnn_model.h5')
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [35]:
model.get_weights()

[array([[-0.02399438,  0.01354884,  0.066531  , ..., -0.02297153,
          0.00197783,  0.04496226],
        [-0.03236635,  0.02194656,  0.05135894, ...,  0.01298204,
         -0.0094652 , -0.01548245],
        [-0.01835768, -0.04338934, -0.00284059, ...,  0.00217873,
         -0.04410315,  0.01969799],
        ...,
        [ 0.05434644, -0.02559111, -0.00467756, ..., -0.0321386 ,
          0.02048911, -0.01425335],
        [ 0.05436032,  0.02770581, -0.06457186, ...,  0.03215596,
         -0.01435175, -0.04247245],
        [ 0.00872363, -0.02753903, -0.02594378, ..., -0.08938783,
          0.03614621, -0.00104547]], shape=(10000, 128), dtype=float32),
 array([[ 0.03144435, -0.25549278,  0.21933821, ..., -0.14744677,
         -0.19887008, -0.19917643],
        [-0.03352899,  0.03743433,  0.24656686, ...,  0.05197809,
         -0.02885623,  0.02184043],
        [-0.07627267,  0.16663401,  0.11464268, ...,  0.20390728,
          0.06120143,  0.13516575],
        ...,
        [ 0.0200739

In [53]:
#Function to decode the review 
# decoded_review = ' '.join([rev_word_index.get(i-3, '?') for i in encoded_review])
#Function to encode the review
import re
def pre_process(text):
    # 1. Strip punctuation so "bad!" becomes "bad"
    text = re.sub(r'[^\w\s]', '', text.lower())
    words = text.split()
    
    # 2. Add the +3 shift to match the training data alignment
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
        
    padded_review = pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [54]:
def predict_sentiment(text):
    processed_review = pre_process(text)
    prediction = model.predict(processed_review)[0][0]
    sentiment = 'Positive' if prediction > 0.5 else 'Negative'
    return sentiment, prediction 

In [ ]:
#Example for Prediction
review = "The movie was ."
sentiment, confidence = predict_sentiment(review)
print(f"Review: {review}")
print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Review: The movie was okay okay.
Predicted Sentiment: Negative (Confidence: 0.2044)


In [46]:
# Run this to debug your mapping
from tensorflow.keras.datasets import imdb
word_index = imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

# Let's see what your pre_process is ACTUALLY sending to the model
test_text = "bad"
words = test_text.lower().split()
# Replace this line with EXACTLY what is in your current pre_process function:
encoded = [word_index.get(w, 2)+3 for w in words] 

print(f"Word: 'bad' -> Index sent to model: {encoded}")
print(f"But in the dictionary, index {encoded[0]} is actually: '{reverse_word_index.get(encoded[0]-3, 'UNKNOWN')}'")


Word: 'bad' -> Index sent to model: [78]
But in the dictionary, index 78 is actually: 'bad'
